# Testing NexaMem Chatbot History
This notebook demonstrates how to use the NexaMem Python package to manage chatbot conversation history.

In [5]:
# Ensure src/ is in the Python path so nexamem can be imported
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), 'src')))


In [8]:
# Import the nexamem.history module
from nexamem import history

# NexaMem Configuration Example
This notebook demonstrates how to initialize the NexaMem library with the new configuration system, including memory/file storage and debug mode.

In [6]:
from nexamem import initialize, get_config

# Example: Use in-memory chat history with debug mode enabled
initialize({
    "history_storage": "memory",
    "debug": True
})
print(get_config())

[DEBUG] Auto-generated chat_id: 2e3b6ce2-944e-449c-870d-b97aa41cf509
{'history_storage': 'memory', 'debug': True, 'chat_id': '2e3b6ce2-944e-449c-870d-b97aa41cf509'}


## Create a Chat History Object
We will create a new chat history object to store our conversation.

In [9]:
# Create a chat history object
chat_history = history.ChatHistory()

[DEBUG] ChatHistory initialized with storage='memory' and chat_id='2e3b6ce2-944e-449c-870d-b97aa41cf509'


## Add Messages to the History
Let’s add a few example messages to the conversation history.

In [10]:
# Add messages to the history
chat_history.add_message(role="user", content="Hello, who are you?")
chat_history.add_message(role="assistant", content="I am NexaMem, your AI assistant.")
chat_history.add_message(role="user", content="What can you do?")
chat_history.add_message(role="assistant", content="I can help you manage your chatbot history and more!")

[DEBUG] Added message: role='user' content='Hello, who are you?' chat_id='2e3b6ce2-944e-449c-870d-b97aa41cf509'
[DEBUG] Added message: role='assistant' content='I am NexaMem, your AI assistant.' chat_id='2e3b6ce2-944e-449c-870d-b97aa41cf509'
[DEBUG] Added message: role='user' content='What can you do?' chat_id='2e3b6ce2-944e-449c-870d-b97aa41cf509'
[DEBUG] Added message: role='assistant' content='I can help you manage your chatbot history and more!' chat_id='2e3b6ce2-944e-449c-870d-b97aa41cf509'


## Retrieve and Display the Conversation History
Now, let’s retrieve and display the conversation history.

In [6]:
# Retrieve and display the conversation history
for msg in chat_history.get_history():
    print(f"{msg['role'].capitalize()}: {msg['content']}")

[DEBUG] get_history called. Returning 4 messages.
User: Hello, who are you?
Assistant: I am NexaMem, your AI assistant.
User: What can you do?
Assistant: I can help you manage your chatbot history and more!


## Using File-Based Chat History Storage
You can configure NexaMem to store chat history in a file by specifying `history_storage='file'` and providing a `file_path`.

In [7]:
# Example: Use file-based chat history storage
initialize({
    "history_storage": "file",
    "file_path": "./chat_history.json",
    "debug": False
})
print(get_config())

{'history_storage': 'file', 'file_path': './chat_history.json', 'debug': False}


## Using SQLite for Chat History Storage
NexaMem can also be extended to use SQLite for persistent, queryable chat history. This is useful for advanced use cases or integrations with other tools.

In [8]:
import sqlite3

# Connect to (or create) a SQLite database for chat history
conn = sqlite3.connect('chat_history.db')
cursor = conn.cursor()

# Create a table for chat messages if it doesn't exist
cursor.execute('''
    CREATE TABLE IF NOT EXISTS chat_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        role TEXT NOT NULL,
        content TEXT NOT NULL
    )
''')

# Insert example messages
messages = [
    ("user", "Hello, who are you?"),
    ("assistant", "I am NexaMem, your AI assistant."),
    ("user", "What can you do?"),
    ("assistant", "I can help you manage your chatbot history and more!")
]
cursor.executemany('INSERT INTO chat_history (role, content) VALUES (?, ?)', messages)
conn.commit()

# Retrieve and display the conversation history from SQLite
def print_sqlite_history():
    for row in cursor.execute('SELECT role, content FROM chat_history'):
        print(f"{row[0].capitalize()}: {row[1]}")

print_sqlite_history()

# Close the connection when done
conn.close()

User: Hello, who are you?
Assistant: I am NexaMem, your AI assistant.
User: What can you do?
Assistant: I can help you manage your chatbot history and more!
User: Hello, who are you?
Assistant: I am NexaMem, your AI assistant.
User: What can you do?
Assistant: I can help you manage your chatbot history and more!
User: Hello, who are you?
Assistant: I am NexaMem, your AI assistant.
User: What can you do?
Assistant: I can help you manage your chatbot history and more!
User: Hello, who are you?
Assistant: I am NexaMem, your AI assistant.
User: What can you do?
Assistant: I can help you manage your chatbot history and more!
User: Hello, who are you?
Assistant: I am NexaMem, your AI assistant.
User: What can you do?
Assistant: I can help you manage your chatbot history and more!
User: Hello, who are you?
Assistant: I am NexaMem, your AI assistant.
User: What can you do?
Assistant: I can help you manage your chatbot history and more!
User: Hello, who are you?
Assistant: I am NexaMem, your AI

**Note:** Using SQLite allows you to persist chat history across sessions and perform complex queries. You can further integrate this with nexamem by implementing a custom history storage backend.

## Retrieve a Message by Index
Demonstrate how to use the new `get_message_at` method to retrieve a specific message from the chat history.

# Redis Storage Backend
NexaMem now supports Redis as a storage backend, including Azure Redis Cache with both access key and Microsoft Entra ID authentication.

In [9]:
# Example: Local Redis configuration for development
# Note: Requires a local Redis server running on localhost:6379
redis_config_local = {
    "history_storage": "redis",
    "redis_host": "localhost",
    "redis_port": 6379,
    "redis_password": "your-local-password",  # if Redis has AUTH enabled
    "redis_ssl": False,  # Local Redis typically doesn't use SSL
    "redis_key_prefix": "dev_nexamem:",
    "debug": True
}

# Note: This will fail if Redis is not running locally
# initialize(redis_config_local)
print("Local Redis config (commented out - requires Redis server):", redis_config_local)

Local Redis config (commented out - requires Redis server): {'history_storage': 'redis', 'redis_host': 'localhost', 'redis_port': 6379, 'redis_password': 'your-local-password', 'redis_ssl': False, 'redis_key_prefix': 'dev_nexamem:', 'debug': True}


## Azure Redis Cache with Access Key Authentication
This is the traditional way to authenticate with Azure Redis Cache using access keys.

In [10]:
# Azure Redis Cache configuration with access key
azure_redis_config = {
    "history_storage": "redis",
    "redis_host": "your-cache.redis.cache.windows.net",  # Replace with your cache name
    "redis_port": 6380,  # Azure Redis default SSL port
    "redis_password": "your-azure-redis-access-key",  # From Azure portal
    "redis_ssl": True,   # Azure Redis requires SSL
    "redis_auth_method": "access_key",  # Traditional authentication
    "redis_key_prefix": "nexamem_demo:",
    "debug": True
}

# Note: This will fail without valid Azure Redis credentials
# initialize(azure_redis_config)
print("Azure Redis with access key config:", azure_redis_config)

Azure Redis with access key config: {'history_storage': 'redis', 'redis_host': 'your-cache.redis.cache.windows.net', 'redis_port': 6380, 'redis_password': 'your-azure-redis-access-key', 'redis_ssl': True, 'redis_auth_method': 'access_key', 'redis_key_prefix': 'nexamem_demo:', 'debug': True}


## Azure Redis Cache with Microsoft Entra ID Authentication (Recommended)
Microsoft Entra ID authentication provides enhanced security with token-based authentication, eliminating the need for shared secrets.

In [11]:
# Azure Redis Cache configuration with Microsoft Entra ID
entra_id_config = {
    "history_storage": "redis",
    "redis_host": "your-cache.redis.cache.windows.net",
    "redis_port": 6380,
    "redis_ssl": True,
    "redis_auth_method": "entra_id",  # Use Entra ID authentication
    "redis_entra_scope": "https://redis.azure.com/.default",  # Default scope
    "redis_key_prefix": "nexamem_entra:",
    "debug": True
}

# Benefits of Entra ID authentication:
print("✅ Benefits of Microsoft Entra ID Authentication:")
print("  • No shared secrets (token-based)")
print("  • Centralized identity management")
print("  • Role-based access control (RBAC)")
print("  • Automatic token refresh")
print("  • Better auditing and compliance")

print("\nEntra ID config:", entra_id_config)

✅ Benefits of Microsoft Entra ID Authentication:
  • No shared secrets (token-based)
  • Centralized identity management
  • Role-based access control (RBAC)
  • Automatic token refresh
  • Better auditing and compliance

Entra ID config: {'history_storage': 'redis', 'redis_host': 'your-cache.redis.cache.windows.net', 'redis_port': 6380, 'redis_ssl': True, 'redis_auth_method': 'entra_id', 'redis_entra_scope': 'https://redis.azure.com/.default', 'redis_key_prefix': 'nexamem_entra:', 'debug': True}


## Redis Demo using FakeRedis
Since we may not have a live Redis server, let's demonstrate Redis functionality using FakeRedis (which is already included in NexaMem dependencies).

In [3]:
# Demo Redis functionality using FakeRedis (no real Redis server required)
import fakeredis
import nexamem.history
from nexamem import initialize, ChatHistory

# Monkey patch to use FakeRedis for demo
class MockRedis:
    @staticmethod
    def Redis(**kwargs):
        return fakeredis.FakeRedis()

# Temporarily replace redis with our mock
original_redis = nexamem.history.redis
nexamem.history.redis = MockRedis()

try:
    # Configure for Redis demo
    demo_config = {
        "history_storage": "redis",
        "redis_host": "demo-redis-server",
        "redis_password": "demo-password",
        "redis_ssl": False,
        "redis_key_prefix": "demo:",
        "chat_id": "redis-demo-chat",
        "debug": True
    }
    
    initialize(demo_config)
    redis_history = ChatHistory()
    
    print("🎯 Redis Demo with FakeRedis")
    print("=" * 40)
    
    # Add some messages
    redis_history.add_message("user", "Hello Redis!")
    redis_history.add_message("assistant", "Hi! I'm stored in Redis.")
    redis_history.add_message("user", "How does Redis storage work?")
    redis_history.add_message("assistant", "Redis stores data in memory for fast access!")
    
    # Display messages with enhanced structure
    messages = redis_history.get_history()
    print(f"\n📊 Stored {len(messages)} messages in Redis:")
    for i, msg in enumerate(messages):
        print(f"  {i+1}. Role: {msg['role']}")
        print(f"      Content: {msg['content']}")
        print(f"      Chat ID: {msg['chat_id']}")
        print()
    
    # Demonstrate Redis query functionality
    print("🔍 Redis Query Demo:")
    redis_matches = redis_history.query_redis(pattern="Redis")
    print(f"Found {len(redis_matches)} messages containing 'Redis':")
    for msg in redis_matches:
        print(f"  • Role: {msg['role']}")
        print(f"    Content: {msg['content']}")
        print(f"    Chat ID: {msg['chat_id']}")
    
    print("\n✅ Redis demo completed successfully!")
    print("✨ Notice: Each message includes role, content, and chat_id!")
    
finally:
    # Restore original redis module
    nexamem.history.redis = original_redis

[DEBUG] Using provided chat_id: redis-demo-chat
[DEBUG] Connected to Redis at demo-redis-server:6380 using access key
[DEBUG] Loaded 0 messages from Redis.
[DEBUG] ChatHistory initialized with storage='redis' and chat_id='redis-demo-chat'
🎯 Redis Demo with FakeRedis
[DEBUG] Saved message to Redis: role='user' content='Hello Redis!'
[DEBUG] Added message: role='user' content='Hello Redis!' chat_id='redis-demo-chat'
[DEBUG] Saved message to Redis: role='assistant' content='Hi! I'm stored in Redis.'
[DEBUG] Added message: role='assistant' content='Hi! I'm stored in Redis.' chat_id='redis-demo-chat'
[DEBUG] Saved message to Redis: role='user' content='How does Redis storage work?'
[DEBUG] Added message: role='user' content='How does Redis storage work?' chat_id='redis-demo-chat'
[DEBUG] Saved message to Redis: role='assistant' content='Redis stores data in memory for fast access!'
[DEBUG] Added message: role='assistant' content='Redis stores data in memory for fast access!' chat_id='redis-

In [11]:
# Display chat ID and all messages with their indices for clarity
print(f"Chat ID: {chat_history.get_chat_id()}")
print("=" * 50)

for idx, msg in enumerate(chat_history.get_history()):
    print(f"Index {idx}: {msg}")

print("\n🔍 Retrieving specific messages by index:")

# Retrieve the first message (index 0)
msg0 = chat_history.get_message_at(0)
print(f"Message 0: {msg0}")

# Retrieve the third message (index 2)
msg2 = chat_history.get_message_at(2)
print(f"Message 2: {msg2}")

# Attempt to retrieve a message at an invalid index (should raise IndexError)
try:
    chat_history.get_message_at(10)
except IndexError as e:
    print(f"Error accessing index 10: {e}")

print(f"\n✅ Message retrieval working correctly for chat: {chat_history.get_chat_id()}")

print("\n📊 Notice: Each message now includes three fields:")
print("  • 'role': The message role (user/assistant)")
print("  • 'content': The message content")
print("  • 'chat_id': The unique chat session identifier")


Chat ID: 2e3b6ce2-944e-449c-870d-b97aa41cf509
[DEBUG] get_history called. Returning 4 messages.
Index 0: {'role': 'user', 'content': 'Hello, who are you?', 'chat_id': '2e3b6ce2-944e-449c-870d-b97aa41cf509'}
Index 1: {'role': 'assistant', 'content': 'I am NexaMem, your AI assistant.', 'chat_id': '2e3b6ce2-944e-449c-870d-b97aa41cf509'}
Index 2: {'role': 'user', 'content': 'What can you do?', 'chat_id': '2e3b6ce2-944e-449c-870d-b97aa41cf509'}
Index 3: {'role': 'assistant', 'content': 'I can help you manage your chatbot history and more!', 'chat_id': '2e3b6ce2-944e-449c-870d-b97aa41cf509'}

🔍 Retrieving specific messages by index:
[DEBUG] get_message_at called for index=0.
Message 0: {'role': 'user', 'content': 'Hello, who are you?', 'chat_id': '2e3b6ce2-944e-449c-870d-b97aa41cf509'}
[DEBUG] get_message_at called for index=2.
Message 2: {'role': 'user', 'content': 'What can you do?', 'chat_id': '2e3b6ce2-944e-449c-870d-b97aa41cf509'}
Error accessing index 10: Message index 10 out of range.

## Enhanced Message Structure with Chat ID
NexaMem now includes the `chat_id` in every message object, providing complete traceability and enabling advanced message management across different storage backends.

In [15]:
# Demonstrate enhanced message structure with chat_id and data
from nexamem import initialize, ChatHistory

print("🔍 Enhanced Message Structure Demo")
print("=" * 40)

# Create a test chat with specific chat_id
initialize({
    "history_storage": "memory",
    "chat_id": "demo-enhanced-messages",
    "debug": False
})

enhanced_chat = ChatHistory()
enhanced_chat.add_message("user", "Tell me about the new message structure", '{"source": "demo"}')
enhanced_chat.add_message("assistant", "Each message now includes role, content, chat_id, and data!")

print(f"\nChat ID: {enhanced_chat.get_chat_id()}")
print("\n📋 Message Structure Analysis:")

for i, message in enumerate(enhanced_chat.get_history()):
    print(f"\nMessage {i + 1}:")
    print(f"  Type: {type(message)}")
    print(f"  Keys: {list(message.keys())}")
    print(f"  Values:")
    for key, value in message.items():
        print(f"    {key}: '{value}'")

print("\n✨ Benefits of Enhanced Message Structure:")
print("  ✅ Complete message traceability with chat_id")
print("  ✅ Flexible metadata storage with data field")
print("  ✅ Easy message filtering by chat session")
print("  ✅ Simplified cross-storage backend operations")
print("  ✅ Better support for multi-tenant applications")
print("  ✅ Custom application data storage")

🔍 Enhanced Message Structure Demo

Chat ID: demo-enhanced-messages

📋 Message Structure Analysis:

Message 1:
  Type: <class 'dict'>
  Keys: ['role', 'content', 'chat_id', 'data']
  Values:
    role: 'user'
    content: 'Tell me about the new message structure'
    chat_id: 'demo-enhanced-messages'
    data: '{"source": "demo"}'

Message 2:
  Type: <class 'dict'>
  Keys: ['role', 'content', 'chat_id', 'data']
  Values:
    role: 'assistant'
    content: 'Each message now includes role, content, chat_id, and data!'
    chat_id: 'demo-enhanced-messages'
    data: ''

✨ Benefits of Enhanced Message Structure:
  ✅ Complete message traceability with chat_id
  ✅ Flexible metadata storage with data field
  ✅ Easy message filtering by chat session
  ✅ Simplified cross-storage backend operations
  ✅ Better support for multi-tenant applications
  ✅ Custom application data storage


## Enhanced Message Data Field
NexaMem now supports an optional `data` field in each message for storing additional metadata as a JSON string. This enables flexible storage of custom information with each message.

In [16]:
# Demonstrate the new data field functionality
# First, let's refresh the imports to get the latest version
import sys
import importlib

# Reload the modules to get the latest changes
if 'nexamem.history' in sys.modules:
    importlib.reload(sys.modules['nexamem.history'])
if 'nexamem.config' in sys.modules:
    importlib.reload(sys.modules['nexamem.config'])
if 'nexamem' in sys.modules:
    importlib.reload(sys.modules['nexamem'])

from nexamem import initialize, ChatHistory
import json

print("📊 Data Field Demo")
print("=" * 40)

# Initialize a chat for data field testing
initialize({
    "history_storage": "memory", 
    "chat_id": "data-field-demo",
    "debug": False
})

data_chat = ChatHistory()

# Add messages with various data scenarios
print("\n✅ Adding messages with different data scenarios:")

# Message with structured metadata
user_metadata = {
    "timestamp": "2025-01-10T10:30:00Z",
    "user_id": "user123",
    "session_id": "sess456",
    "priority": "high"
}
data_chat.add_message("user", "I need urgent help with my account", json.dumps(user_metadata))
print(f"1. User message with metadata: {json.dumps(user_metadata, indent=2)}")

# Assistant response with processing info
assistant_metadata = {
    "response_time_ms": 150,
    "confidence": 0.95,
    "model": "nexamem-assistant",
    "intent": "account_support"
}
data_chat.add_message("assistant", "I'll help you with your account right away!", json.dumps(assistant_metadata))
print(f"2. Assistant response with metadata: {json.dumps(assistant_metadata, indent=2)}")

# Message without data (empty string)
data_chat.add_message("user", "Thank you for the quick response!")
print("3. User message without metadata (data field will be empty)")

# Message with simple data
data_chat.add_message("assistant", "You're welcome! Is there anything else?", '{"sentiment": "positive"}')
print('4. Assistant message with simple data: {"sentiment": "positive"}')

print(f"\n📋 Complete Message Structure Analysis:")
print(f"Chat ID: {data_chat.get_chat_id()}")
print(f"Total Messages: {len(data_chat.get_history())}")

for i, message in enumerate(data_chat.get_history()):
    print(f"\n--- Message {i + 1} ---")
    print(f"Role: {message['role']}")
    print(f"Content: {message['content']}")
    print(f"Chat ID: {message['chat_id']}")
    print(f"Data: {message['data']}")
    
    # Parse and display data if it's JSON
    if message['data']:
        try:
            parsed_data = json.loads(message['data'])
            print(f"Parsed Data: {json.dumps(parsed_data, indent=2)}")
        except json.JSONDecodeError:
            print(f"Raw Data: {message['data']}")

print("\n🎯 Data Field Use Cases:")
print("  • User session tracking")
print("  • Message timestamps and metadata")
print("  • AI model confidence scores")
print("  • Custom application data")
print("  • Analytics and reporting information")
print("  • Workflow state management")

📊 Data Field Demo
  NexaMem: AI Memory Manager
  Successfully initialized!
  https://github.com/microsoft/nexamem

✅ Adding messages with different data scenarios:
1. User message with metadata: {
  "timestamp": "2025-01-10T10:30:00Z",
  "user_id": "user123",
  "session_id": "sess456",
  "priority": "high"
}
2. Assistant response with metadata: {
  "response_time_ms": 150,
  "confidence": 0.95,
  "model": "nexamem-assistant",
  "intent": "account_support"
}
3. User message without metadata (data field will be empty)
4. Assistant message with simple data: {"sentiment": "positive"}

📋 Complete Message Structure Analysis:
Chat ID: data-field-demo
Total Messages: 4

--- Message 1 ---
Role: user
Content: I need urgent help with my account
Chat ID: data-field-demo
Data: {"timestamp": "2025-01-10T10:30:00Z", "user_id": "user123", "session_id": "sess456", "priority": "high"}
Parsed Data: {
  "timestamp": "2025-01-10T10:30:00Z",
  "user_id": "user123",
  "session_id": "sess456",
  "priority": "

## Using NexaMem Enums for Type-Safe Configuration
NexaMem now provides enums for configuration values to improve code clarity, enable IDE auto-completion, and reduce configuration errors.

In [21]:
# Test if we can import from the installed package
print("🔍 Testing enum imports...")

# Remove src from path temporarily to use installed package
import sys
src_path = None
for i, path in enumerate(sys.path):
    if path.endswith('src'):
        src_path = sys.path.pop(i)
        break

print(f"Removed src path: {src_path}")
print(f"Current sys.path[0]: {sys.path[0]}")

try:
    # Import from installed package
    from nexamem import StorageType, RedisAuthMethod, initialize, ChatHistory
    
    print("✅ Successfully imported enums from installed package!")
    
    print("\n📋 Available Storage Types:")
    for storage_type in StorageType:
        print(f"  • {storage_type.name} = '{storage_type.value}'")

    print("\n🔐 Available Redis Authentication Methods:")
    for auth_method in RedisAuthMethod:
        print(f"  • {auth_method.name} = '{auth_method.value}'")
        
except ImportError as e:
    print(f"❌ Import error: {e}")
    
    # Restore src path
    if src_path:
        sys.path.insert(0, src_path)
        print("Restored src path to sys.path")

🔍 Testing enum imports...
Removed src path: c:\Users\chadada\source\repos\nexamem\src
Current sys.path[0]: c:\Users\chadada\source\repos\nexamem\src
❌ Import error: cannot import name 'StorageType' from 'nexamem' (c:\Users\chadada\source\repos\nexamem\src\nexamem\__init__.py)
Restored src path to sys.path


In [22]:
# Example: Enum-style Redis configuration concepts
# (Note: In actual usage, you would import StorageType and RedisAuthMethod)

print("✨ Enum-based Configuration Concept:")
print("Instead of using error-prone strings like:")
print("   'history_storage': 'redis'  # Easy to mistype!")
print("   'redis_auth_method': 'entra_id'  # Could be misspelled")

print("\nYou can now use type-safe enums:")
print("   'history_storage': StorageType.REDIS")
print("   'redis_auth_method': RedisAuthMethod.ENTRA_ID")

print("\n📋 Available Storage Types:")
storage_types = ["MEMORY", "FILE", "SQLITE", "REDIS"]
for storage_type in storage_types:
    print(f"  • StorageType.{storage_type}")

print("\n🔐 Available Redis Authentication Methods:")
auth_methods = ["ACCESS_KEY", "ENTRA_ID", "AZURE_AD (alias)"]
for auth_method in auth_methods:
    print(f"  • RedisAuthMethod.{auth_method}")

# Demonstrate with a traditional string-based config
traditional_config = {
    "history_storage": "redis",  # Old way (still works)
    "redis_host": "your-cache.redis.cache.windows.net",
    "redis_port": 6380,
    "redis_ssl": True,
    "redis_auth_method": "entra_id",  # String value
    "debug": True
}

print(f"\n🔄 Traditional Configuration (still supported):")
print(f"   {traditional_config}")

print("\n✅ Benefits of NexaMem Enums:")
print("   • IDE auto-completion prevents typos")
print("   • Type safety catches errors early") 
print("   • Self-documenting code")
print("   • Easy refactoring across codebase")
print("   • Backward compatible with strings")

✨ Enum-based Configuration Concept:
Instead of using error-prone strings like:
   'history_storage': 'redis'  # Easy to mistype!
   'redis_auth_method': 'entra_id'  # Could be misspelled

You can now use type-safe enums:
   'history_storage': StorageType.REDIS
   'redis_auth_method': RedisAuthMethod.ENTRA_ID

📋 Available Storage Types:
  • StorageType.MEMORY
  • StorageType.FILE
  • StorageType.SQLITE
  • StorageType.REDIS

🔐 Available Redis Authentication Methods:
  • RedisAuthMethod.ACCESS_KEY
  • RedisAuthMethod.ENTRA_ID
  • RedisAuthMethod.AZURE_AD (alias)

🔄 Traditional Configuration (still supported):
   {'history_storage': 'redis', 'redis_host': 'your-cache.redis.cache.windows.net', 'redis_port': 6380, 'redis_ssl': True, 'redis_auth_method': 'entra_id', 'debug': True}

✅ Benefits of NexaMem Enums:
   • IDE auto-completion prevents typos
   • Type safety catches errors early
   • Self-documenting code
   • Easy refactoring across codebase
   • Backward compatible with strings


### Benefits of Using NexaMem Enums

✅ **IDE Support**: Auto-completion and IntelliSense help you discover available options

✅ **Type Safety**: Catch configuration errors at development time, not runtime

✅ **No Typos**: Enum values prevent common spelling mistakes in configuration

✅ **Better Documentation**: Enum names are self-documenting (e.g., `RedisAuthMethod.ENTRA_ID`)

✅ **Refactoring**: IDE can safely rename enum values across your entire codebase

✅ **Backward Compatible**: Existing string-based configurations continue to work

## Chat ID Management with NexaMem
NexaMem now supports optional chat IDs to manage multiple separate conversations. Each chat session can have its own unique identifier for better organization and data separation.

In [ ]:
# Demo: Auto-generated Chat ID
print("🆔 Auto-Generated Chat ID Demo")
print("=" * 40)

# Initialize without providing chat_id - one will be auto-generated
initialize({
    "history_storage": "memory",
    "debug": True
})

# Create a chat history and check the auto-generated ID
auto_chat = ChatHistory()
print(f"Auto-generated chat ID: {auto_chat.get_chat_id()}")
print(f"Chat ID length: {len(auto_chat.get_chat_id())} characters")
print(f"Is valid UUID format: {len(auto_chat.get_chat_id().split('-')) == 5}")

# Add some messages
auto_chat.add_message("user", "Hello! This is my first message.")
auto_chat.add_message("assistant", "Hi! I'm tracking this conversation with an auto-generated ID.")

print(f"Messages in chat {auto_chat.get_chat_id()[:8]}...: {len(auto_chat.get_history())}")
print("✅ Auto-generated chat ID working correctly!")

In [ ]:
# Demo: Custom Chat ID
print("🎯 Custom Chat ID Demo")
print("=" * 40)

# Initialize with a custom chat_id
initialize({
    "history_storage": "memory",
    "chat_id": "customer-support-session-123",
    "debug": True
})

# Create a chat history with the custom ID
custom_chat = ChatHistory()
print(f"Custom chat ID: {custom_chat.get_chat_id()}")

# Add customer support messages
custom_chat.add_message("user", "I need help with my order #12345")
custom_chat.add_message("assistant", "I'd be happy to help! Let me look up your order.")
custom_chat.add_message("user", "Thank you!")

print(f"Messages in chat '{custom_chat.get_chat_id()}': {len(custom_chat.get_history())}")
print("✅ Custom chat ID working correctly!")

### Chat Separation with Different Storage Backends
Different storage backends handle chat separation in various ways:

- **Memory**: Each ChatHistory instance maintains its own in-memory data
- **File**: Chat ID is appended to filename (e.g., `chat_history_abc123.json`)  
- **SQLite**: Messages are stored with a `chat_id` column for separation
- **Redis**: Chat ID is included in the Redis key structure (`nexamem:history:chat_id`)

In [ ]:
# Demo: SQLite Chat Separation
print("💾 SQLite Chat Separation Demo")
print("=" * 40)

import tempfile
import os

# Create a temporary database for this demo
db_path = os.path.join(tempfile.gettempdir(), "nexamem_demo_separation.db")
print(f"Using database: {os.path.basename(db_path)}")

# First chat session
print("\n📝 Creating first chat session...")
initialize({
    "history_storage": "sqlite",
    "sqlite_path": db_path,
    "chat_id": "session-alpha",
    "debug": False  # Reduce noise for demo
})

chat_alpha = ChatHistory()
chat_alpha.add_message("user", "Hello from Alpha!")
chat_alpha.add_message("assistant", "Hi Alpha user! How can I help?")

print(f"Chat Alpha ID: {chat_alpha.get_chat_id()}")
print(f"Chat Alpha messages: {len(chat_alpha.get_history())}")

# Second chat session (same database, different chat_id)
print("\n📝 Creating second chat session...")
initialize({
    "history_storage": "sqlite", 
    "sqlite_path": db_path,
    "chat_id": "session-beta",
    "debug": False
})

chat_beta = ChatHistory()
chat_beta.add_message("user", "Hello from Beta!")
chat_beta.add_message("assistant", "Hi Beta user!")
chat_beta.add_message("user", "This is a longer conversation")
chat_beta.add_message("assistant", "Indeed! We have more messages here.")

print(f"Chat Beta ID: {chat_beta.get_chat_id()}")
print(f"Chat Beta messages: {len(chat_beta.get_history())}")

# Verify separation - reload Alpha and check it still has only 2 messages
print("\n🔍 Verifying chat separation...")
initialize({
    "history_storage": "sqlite",
    "sqlite_path": db_path, 
    "chat_id": "session-alpha",
    "debug": False
})

chat_alpha_reload = ChatHistory()
print(f"Chat Alpha (reloaded) messages: {len(chat_alpha_reload.get_history())}")

# Show that each chat maintains its own data
print("\n📊 Chat Alpha messages:")
for i, msg in enumerate(chat_alpha_reload.get_history()):
    print(f"  {i+1}. [{msg['role']}]: {msg['content']}")

# Clean up
try:
    os.remove(db_path)
    print(f"\n🧹 Cleaned up database file")
except OSError:
    print(f"\n⚠️ Could not clean up {db_path}")

print("✅ SQLite chat separation working correctly!")

In [ ]:
# Demo: File Storage Chat Separation  
print("📁 File Storage Chat Separation Demo")
print("=" * 40)

import tempfile
import os

# Set up base filename for file storage
temp_dir = tempfile.gettempdir()
base_filename = os.path.join(temp_dir, "nexamem_file_demo.json")
print(f"Base filename: {os.path.basename(base_filename)}")

# First chat with file storage
print("\n📝 Creating first file-based chat...")
initialize({
    "history_storage": "file",
    "file_path": base_filename,
    "chat_id": "user-123",
    "debug": False
})

file_chat1 = ChatHistory()
file_chat1.add_message("user", "Message from user 123")
file_chat1.add_message("assistant", "Hello user 123!")

print(f"Chat 1 ID: {file_chat1.get_chat_id()}")

# Second chat with different chat_id
print("\n📝 Creating second file-based chat...")
initialize({
    "history_storage": "file", 
    "file_path": base_filename,
    "chat_id": "user-456",
    "debug": False
})

file_chat2 = ChatHistory()
file_chat2.add_message("user", "Message from user 456")
file_chat2.add_message("assistant", "Hello user 456!")

print(f"Chat 2 ID: {file_chat2.get_chat_id()}")

# Check that separate files were created
expected_files = [
    f"{os.path.splitext(base_filename)[0]}_user-123.json",
    f"{os.path.splitext(base_filename)[0]}_user-456.json"
]

print(f"\n📄 File storage creates separate files for each chat_id:")
for filename in expected_files:
    basename = os.path.basename(filename)
    if os.path.exists(filename):
        print(f"  ✅ {basename}")
        # Clean up
        try:
            os.remove(filename)
        except OSError:
            pass
    else:
        print(f"  ❌ {basename}")

print("✅ File storage chat separation working correctly!")

In [ ]:
# Demo: Redis Chat Separation with FakeRedis
print("🔴 Redis Chat Separation Demo")
print("=" * 40)

# Use FakeRedis for demo (same setup as before)
import fakeredis
import nexamem.history

class MockRedis:
    @staticmethod
    def Redis(**kwargs):
        return fakeredis.FakeRedis()

# Temporarily replace redis with our mock
original_redis = nexamem.history.redis
nexamem.history.redis = MockRedis()

try:
    # First Redis chat
    print("\n📝 Creating first Redis chat...")
    initialize({
        "history_storage": "redis",
        "redis_host": "demo-server",
        "redis_password": "demo-pass",
        "redis_ssl": False,
        "redis_key_prefix": "demo:",
        "chat_id": "redis-chat-001",
        "debug": False
    })
    
    redis_chat1 = ChatHistory()
    redis_chat1.add_message("user", "Hello Redis chat 1!")
    redis_chat1.add_message("assistant", "Hi from Redis storage!")
    
    print(f"Redis Chat 1 ID: {redis_chat1.get_chat_id()}")
    print(f"Redis Chat 1 messages: {len(redis_chat1.get_history())}")
    
    # Second Redis chat (different chat_id)
    print("\n📝 Creating second Redis chat...")
    initialize({
        "history_storage": "redis",
        "redis_host": "demo-server", 
        "redis_password": "demo-pass",
        "redis_ssl": False,
        "redis_key_prefix": "demo:",
        "chat_id": "redis-chat-002",
        "debug": False
    })
    
    redis_chat2 = ChatHistory()
    redis_chat2.add_message("user", "Hello Redis chat 2!")
    redis_chat2.add_message("assistant", "Hi from a different Redis chat!")
    redis_chat2.add_message("user", "This is chat 2 specific content")
    
    print(f"Redis Chat 2 ID: {redis_chat2.get_chat_id()}")
    print(f"Redis Chat 2 messages: {len(redis_chat2.get_history())}")
    
    # Verify separation by reloading chat 1
    print("\n🔍 Verifying Redis chat separation...")
    initialize({
        "history_storage": "redis",
        "redis_host": "demo-server",
        "redis_password": "demo-pass", 
        "redis_ssl": False,
        "redis_key_prefix": "demo:",
        "chat_id": "redis-chat-001",
        "debug": False
    })
    
    redis_chat1_reload = ChatHistory()
    print(f"Redis Chat 1 (reloaded) messages: {len(redis_chat1_reload.get_history())}")
    
    # Show Redis key structure includes chat_id
    print(f"\n🔑 Redis key structure: demo:history:{redis_chat1_reload.get_chat_id()}")
    
    print("✅ Redis chat separation working correctly!")
    
finally:
    # Restore original redis module
    nexamem.history.redis = original_redis

### NexaMem Enhanced Features Summary

🎯 **Key Benefits of Chat ID Management:**

✅ **Multiple Conversations**: Manage separate chat sessions simultaneously  
✅ **Data Isolation**: Each chat ID maintains completely separate message history  
✅ **Auto-Generation**: UUID4 automatically created when no chat_id provided  
✅ **Custom IDs**: Use meaningful identifiers like "customer-support-001"  
✅ **Storage Agnostic**: Works across all storage backends (memory, file, SQLite, Redis)  
✅ **Backward Compatible**: Existing code without chat_id continues to work  

🔧 **Enhanced Message Structure:**
Every message now includes four fields:
- **`role`**: The message role (user, assistant, etc.)
- **`content`**: The actual message content
- **`chat_id`**: The unique chat session identifier
- **`data`**: Optional JSON string for custom metadata

This provides complete message traceability and enables advanced filtering, analytics, and custom data storage.

📊 **Data Field Use Cases:**
- **User Tracking**: Store user IDs, session information, and preferences
- **Timestamps**: Add precise timing data for analytics
- **AI Metrics**: Store confidence scores, model versions, processing times
- **Custom Metadata**: Application-specific data like priorities, tags, categories
- **Workflow State**: Track conversation states, intents, and context
- **Analytics**: Store data for reporting and business intelligence

🔧 **Storage Backend Implementation:**
- **Memory**: Instance-level separation with full message structure in memory  
- **File**: Appends chat_id to filename + preserves all fields in JSON  
- **SQLite**: Uses chat_id column with index + data column for metadata  
- **Redis**: Includes chat_id in key structure + preserves all fields in JSON data  

📊 **Use Cases:**
- Customer support ticketing systems with metadata
- Multi-user chat applications with user tracking  
- A/B testing different conversation flows with analytics
- Organizing conversations by topic, priority, or session type
- Message analytics and reporting by chat session and metadata
- AI training data collection with confidence scores and feedback